"""
Stage 2 — Noun chunk extraction + referring-expression expansion.
Tuned for PixMo-Cap's long, narration-derived dense captions.

Main safeguards:
  1. Parse each sentence independently so noun chunks cannot cross sentences.
  2. Apostrophes are never quotation delimiters.
  3. Apostrophes in possessives are ignored: "bird's" -> "birds".
  4. Drop bare pronouns.
  5. Strip deictic modifiers from otherwise useful noun chunks.
  6. Treat double-quoted spans as atomic.
  7. Block abstract, evaluative, media-referential and positional metadata.
  8. Expand only through safe nominal modifiers.
  9. Never expand through relative/adverbial/explanatory clauses.
 10. Preserve punctuation and spacing when tokens are removed.
 11. Reject malformed or sentence-crossing expressions.
"""
import argparse
import json
import re
import spacy
from tqdm import tqdm
# ---------------------------------------------------------------------------
# ABSTRACT / NON-VISUAL NOUNS
# ---------------------------------------------------------------------------
ABSTRACT_NOUNS = {
    "aesthetic","atmosphere","ambiance","ambience","beauty","behavior","behaviour","blend","charm","chaos","cleanliness","cohesion","concept","context","courage","depth","demeanor","demeanour","decision","decision-making","elegance","element","emphasis","energy","essence","explanation","feeling","feature","focus","freedom","fusion","happiness","hate","history","honesty","hope","humor","humour","idea","imagination","importance","impression","insight","insights","intention","intensity","joy","justice","life","logic","look","love","meaning","memory","mood","moment","mystique","nature","notion","passion","peace","personality","perspective","presence","purpose","quality","reality","reason","sadness","sense","significance","silence","sophistication","soul","spirit","strategy","strategies","style","synthesis","testament","thought","time","tone","touch","trait","truth","vibe","viewpoint","wisdom","wonder","appearance","characteristic","attention","focus","aspect","purpose",
}
NON_VISUAL_NOUNS = {
    "communication","content","discussion","experience","information","location","overview","process","reference","response","situation","subject","topic","trend","view","explanation","description","caption","narrative","story","article","commentary","message","instruction","option","choice","example","result","effect","reason","cause","analysis","comparison","interpretation","meaning",
}
MEDIA_REFERENTIAL_NOUNS = {
    "image","photo","photograph","picture","screenshot","video","shot","scene","viewer","camera","perspective","angle","frame",
}

# ---------------------------------------------------------------------------
# POSITIONAL / FRAMING METADATA
# ---------------------------------------------------------------------------
# These are blocked when they function as the HEAD of a chunk or as
# metadata-bearing constructions such as "left side", "top portion",
# "upper right corner", etc.
FRAMING_HEADS = {
    "corner","edge","margin","quadrant","quarter","portion","section","side","half","periphery",
}
FRAMING_WORDS = {
    "top","bottom","left","right","upper","lower","center","centre","middle","front","back",
}
FRAMING_CONSTRUCTIONS = {
    ("left","side"),("right","side"),("top","side"),("bottom","side"),
    ("upper","section"),("lower","section"),("top","section"),("bottom","section"),
    ("upper","portion"),("lower","portion"),("top","portion"),("bottom","portion"),
    ("upper","half"),("lower","half"),("top","half"),("bottom","half"),
    ("upper","corner"),("lower","corner"),("top","corner"),("bottom","corner"),
    ("left","corner"),("right","corner"),("left","quadrant"),("right","quadrant"),
    ("upper","quadrant"),("lower","quadrant"),("left","quarter"),("right","quarter"),
    ("upper","quarter"),("lower","quarter"),
}
BLOCKED_HEAD_NOUNS = ABSTRACT_NOUNS | NON_VISUAL_NOUNS | MEDIA_REFERENTIAL_NOUNS | FRAMING_HEADS

# ---------------------------------------------------------------------------
# MODIFIER FILTERS
# ---------------------------------------------------------------------------
DEICTIC_MODIFIERS = {
    "this","that","these","those","other","another","such","same","former","latter","aforementioned",
}
EVALUATIVE_MODIFIERS = {
    "adorable","amazing","appealing","attractive","beautiful","captivating","charming","cute","delightful","elegant","endearing","exquisite","extraordinary","fabulous","fantastic","gorgeous","handsome","impressive","iconic","interesting","intriguing","lovely","magnificent","memorable","mysterious","notable","remarkable","refined","sophisticated","special","spectacular","striking","stunning","superb","unique","wonderful","whimsical",
}
# These are especially dangerous because spaCy can parse them as ordinary
# modifiers even though they introduce caption-writer commentary.
COMMENTARY_MODIFIERS = {
    "reminiscent","evocative","suggestive","indicative","representative","symbolic","metaphorical","nostalgic","relevant","noteworthy",
}
# Words which often describe the caption author's interpretation rather than
# a visible property. Kept separate from EVALUATIVE_MODIFIERS so the filter
# remains easy to tune.
COMMENTARY_HEADS = {
    "resemblance","similarity","comparison","reference","association","connection","interpretation","symbolism","meaning",
}

# ---------------------------------------------------------------------------
# DEPENDENCY RULES
# ---------------------------------------------------------------------------
# Safe dependencies for expanding an object description.
SAFE_MODIFIER_DEPS = {
    "det","amod","compound","nummod","poss","quantmod","npadvmod",
}
# Prepositional constructions are useful, but we only allow a controlled
# subset of prepositions rather than every prep in the sentence.
SAFE_PREPOSITIONS = {
    "of","with","in","on","wearing","holding","carrying","made","covered","filled",
}
CLAUSE_BOUNDARY_DEPS = {
    "relcl","advcl","acl","ccomp","xcomp","parataxis","conj","appos",
}
MAX_REF_EXP_TOKENS = 12
MAX_TEXT_CHARS = 200

# ---------------------------------------------------------------------------
# QUOTES
# ---------------------------------------------------------------------------
# IMPORTANT: single quotes are deliberately absent.
QUOTE_PATTERN = re.compile(r'"([^"]*)"|“([^”]*)”')

def find_quote_spans(text):
    return [(m.start(),m.end()) for m in QUOTE_PATTERN.finditer(text)]

def overlap_kind(span, quote_spans):
    s,e = span
    for qs,qe in quote_spans:
        if qs <= s and e <= qe:
            return "contained",(qs,qe)
        if s < qe and e > qs:
            return "partial",(qs,qe)
    return "none",None

# ---------------------------------------------------------------------------
# SENTENCE SPLITTING
# ---------------------------------------------------------------------------
ABBREVIATIONS = {
    "mr","mrs","ms","dr","prof","sr","jr","st","vs","etc","e.g","i.e","u.s","u.k","a.m","p.m",
}

def split_sentences(text):
    """
    Conservative sentence splitter.
    We only split at sentence punctuation when:
      - we are not inside double quotes
      - the punctuation is not part of a decimal
      - the preceding token is not a common abbreviation
    """
    spans = []
    start = 0
    quote_open = False
    i = 0
    while i < len(text):
        ch = text[i]
        if ch in {'"','“'}:
            quote_open = not quote_open
            i += 1
            continue
        if ch == "”":
            quote_open = False
            i += 1
            continue
        if not quote_open and ch in ".!?":
            end = i + 1
            while end < len(text) and text[end] in ".!?":
                end += 1
            if ch == "." and i > 0 and i + 1 < len(text):
                if text[i - 1].isdigit() and text[i + 1].isdigit():
                    i += 1
                    continue
            prefix = text[max(start,i-12):i]
            match = re.search(r"([A-Za-z.]+)$",prefix)
            word_before = match.group(1).rstrip(".").lower() if match else ""
            if word_before in ABBREVIATIONS:
                i += 1
                continue
            if end < len(text) and not text[end].isspace():
                i += 1
                continue
            spans.append((start,end))
            start = end
            while start < len(text) and text[start].isspace():
                start += 1
            i = start
            continue
        i += 1
    if start < len(text):
        spans.append((start,len(text)))
    return spans

# ---------------------------------------------------------------------------
# BASIC TOKEN HELPERS
# ---------------------------------------------------------------------------
def is_bare_pronoun_chunk(chunk):
    return len(chunk) == 1 and chunk.root.pos_ == "PRON"

def is_deictic(tok):
    return tok.lemma_.lower() in DEICTIC_MODIFIERS

def is_evaluative(tok):
    return tok.lemma_.lower() in EVALUATIVE_MODIFIERS

def is_commentary_modifier(tok):
    return tok.lemma_.lower() in COMMENTARY_MODIFIERS

def is_possessive_apostrophe(tok):
    return tok.text in {"'","’"}

def is_possessive_s(tok,doc):
    if tok.i == 0:
        return False
    prev = doc[tok.i-1]
    return tok.text.lower() == "s" and prev.text in {"'","’"}

# ---------------------------------------------------------------------------
# TEXT RECONSTRUCTION
# ---------------------------------------------------------------------------
def normalize_apostrophes(text):
    """
    User requirement:
      bird's -> birds
      Barrett-Jackson's -> Barrett-Jacksons
    We remove the apostrophe but keep the s.
    """
    text = re.sub(r"['’]s\b","s",text,flags=re.IGNORECASE)
    text = re.sub(r"['’]","",text)
    return text

def clean_spacing(text):
    text = re.sub(r"[ \t]+"," ",text)
    text = re.sub(r"\s+([,.;:!?])",r"\1",text)
    text = re.sub(r"([(\[]) +",r"\1",text)
    text = re.sub(r" +([)\]])",r"\1",text)
    text = re.sub(r"\s*([/])\s*",r"\1",text)
    text = re.sub(r"\s*-\s*","-",text)
    text = normalize_apostrophes(text)
    return text.strip()

def render_tokens(token_idxs,doc):
    """
    Render kept tokens while preserving punctuation correctly.
    If tokens were removed between two lexical tokens, insert a space.
    """
    idxs = sorted(set(token_idxs))
    if not idxs:
        return ""
    parts = []
    for pos,i in enumerate(idxs):
        tok = doc[i]
        if is_possessive_apostrophe(tok):
            continue
        if is_possessive_s(tok,doc):
            parts.append(tok.text)
        else:
            parts.append(tok.text)
        if pos == len(idxs)-1:
            continue
        next_i = idxs[pos+1]
        next_tok = doc[next_i]
        if next_i == i+1:
            if tok.whitespace_:
                parts.append(tok.whitespace_)
            elif not tok.is_punct and not next_tok.is_punct:
                parts.append(" ")
        else:
            if tok.is_punct:
                if tok.text in {",",";","-","–","—","/"}:
                    parts.append(" ")
                elif next_tok.is_punct:
                    pass
                else:
                    parts.append(" ")
            elif next_tok.is_punct:
                pass
            else:
                parts.append(" ")
    return clean_spacing("".join(parts))

# ---------------------------------------------------------------------------
# MALFORMED SPAN CHECKS
# ---------------------------------------------------------------------------
BAD_SPAN_CHARS = re.compile(r"[.!?\n]")

def is_malformed_span(text):
    if not text:
        return True
    if BAD_SPAN_CHARS.search(text):
        return True
    if len(text) > MAX_TEXT_CHARS:
        return True
    if re.search(r"[A-Za-z][,;:][A-Za-z]",text):
        return True
    return False

# ---------------------------------------------------------------------------
# POSITIONAL FILTERING
# ---------------------------------------------------------------------------
def is_framing_metadata(chunk):
    root = chunk.root
    root_lemma = root.lemma_.lower()
    if root_lemma in FRAMING_HEADS:
        return True
    lemmas = [tok.lemma_.lower() for tok in chunk]
    for a,b in FRAMING_CONSTRUCTIONS:
        if a in lemmas and b in lemmas:
            return True
    # "top/bottom/left/right" are only blocked when they are themselves
    # the head or clearly part of a framing construction.
    if root_lemma in {"top","bottom","left","right","center","centre","middle","front","back"}:
        return True
    return False

# ---------------------------------------------------------------------------
# CHUNK FILTERING
# ---------------------------------------------------------------------------
def clean_chunk_tokens(chunk):
    keep = []
    for tok in chunk:
        lemma = tok.lemma_.lower()
        if is_possessive_apostrophe(tok):
            continue
        if is_deictic(tok) and tok.dep_ in {"det","amod"}:
            continue
        if is_evaluative(tok) and tok.dep_ == "amod":
            continue
        if is_commentary_modifier(tok) and tok.dep_ in {"amod","advmod"}:
            continue
        if lemma in COMMENTARY_HEADS and tok == chunk.root:
            return None
        keep.append(tok.i)
    if not keep or chunk.root.i not in keep:
        return None
    return keep

def head_is_blocked(chunk):
    return chunk.root.lemma_.lower() in BLOCKED_HEAD_NOUNS

# ---------------------------------------------------------------------------
# REFERRING-EXPRESSION EXPANSION
# ---------------------------------------------------------------------------
def expand_to_referring_expression(root_tok,base_tokens,doc):
    """
    Expand only through modifiers that describe the same nominal entity.

    Crucially:
      - no relcl / acl / advcl / ccomp / xcomp / parataxis
      - no conj/appos
      - no commentary adjectives
      - no evaluative adjectives
      - only selected prepositions
      - never enter a verb phrase
    """
    selected = set(base_tokens)

    def add_subtree(tok,depth=0):
        if depth > 6:
            return
        for child in tok.children:
            dep = child.dep_
            lemma = child.lemma_.lower()
            if dep in CLAUSE_BOUNDARY_DEPS:
                continue
            if child.pos_ in {"VERB","AUX"}:
                continue
            if is_deictic(child) or is_evaluative(child) or is_commentary_modifier(child):
                continue
            if lemma in BLOCKED_HEAD_NOUNS:
                continue
            if dep in SAFE_MODIFIER_DEPS:
                selected.add(child.i)
                add_subtree(child,depth+1)
                continue
            if dep == "prep":
                if lemma not in SAFE_PREPOSITIONS:
                    continue
                pobj = next((x for x in child.children if x.dep_ in {"pobj","pcomp"}),None)
                if pobj is None:
                    continue
                if pobj.pos_ in {"VERB","AUX"}:
                    continue
                if pobj.lemma_.lower() in BLOCKED_HEAD_NOUNS:
                    continue
                selected.add(child.i)
                selected.add(pobj.i)
                add_subtree(pobj,depth+1)
                continue
            if dep == "pobj":
                if lemma in BLOCKED_HEAD_NOUNS:
                    continue
    add_subtree(root_tok)
    if len(selected) > MAX_REF_EXP_TOKENS:
        return None
    return selected

# ---------------------------------------------------------------------------
# QUOTED ATOMIC SPANS
# ---------------------------------------------------------------------------
def extract_atomic_quote(caption,quote_span):
    qs,qe = quote_span
    text = caption[qs:qe].strip()
    if not text or len(text) > MAX_TEXT_CHARS:
        return None
    return {
        "text":text,
        "start_char":qs,
        "end_char":qe,
        "source_chunk":text,
    }

# ---------------------------------------------------------------------------
# DEDUPLICATION
# ---------------------------------------------------------------------------
def dedupe_contained(expressions):
    """
    Keep the most informative non-contained expressions.
    Exact duplicates are removed.
    """
    unique = []
    seen = set()
    for e in expressions:
        key = (e["text"],e["start_char"],e["end_char"])
        if key in seen:
            continue
        seen.add(key)
        unique.append(e)
    unique.sort(key=lambda e:(e["start_char"],-(e["end_char"]-e["start_char"])))
    kept = []
    for e in unique:
        if any(
            k["start_char"] <= e["start_char"] and
            e["end_char"] <= k["end_char"] and
            (k["start_char"] != e["start_char"] or k["end_char"] != e["end_char"])
            for k in kept
        ):
            continue
        kept.append(e)
    return kept

# ---------------------------------------------------------------------------
# PROCESS ONE SENTENCE
# ---------------------------------------------------------------------------
def process_sentence(nlp,sentence_text,sentence_offset,global_quote_spans):
    doc = nlp(sentence_text)
    local_quotes = []
    for qs,qe in global_quote_spans:
        if qs >= sentence_offset and qe <= sentence_offset + len(sentence_text):
            local_quotes.append((qs-sentence_offset,qe-sentence_offset))
    noun_chunks = []
    ref_exps = []
    seen_quotes = set()

    for chunk in doc.noun_chunks:
        kind,q = overlap_kind((chunk.start_char,chunk.end_char),local_quotes)

        if kind == "partial":
            continue

        if kind == "contained":
            if q in seen_quotes:
                continue
            seen_quotes.add(q)
            atomic = extract_atomic_quote(sentence_text,(q[0],q[1]))
            if atomic is None:
                continue
            atomic["start_char"] += sentence_offset
            atomic["end_char"] += sentence_offset
            noun_chunks.append({
                "text":atomic["text"],
                "start_char":atomic["start_char"],
                "end_char":atomic["end_char"],
            })
            ref_exps.append({
                "text":atomic["text"],
                "start_char":atomic["start_char"],
                "end_char":atomic["end_char"],
                "source_chunk":atomic["text"],
            })
            continue

        if is_bare_pronoun_chunk(chunk):
            continue
        if head_is_blocked(chunk):
            continue
        if is_framing_metadata(chunk):
            continue

        trimmed_idxs = clean_chunk_tokens(chunk)
        if trimmed_idxs is None:
            continue

        text = render_tokens(trimmed_idxs,doc)
        if is_malformed_span(text):
            continue

        noun_chunks.append({
            "text":text,
            "start_char":sentence_offset + chunk.start_char,
            "end_char":sentence_offset + chunk.end_char,
        })

        exp_idxs = expand_to_referring_expression(chunk.root,trimmed_idxs,doc)
        if exp_idxs is None:
            continue

        exp_text = render_tokens(exp_idxs,doc)
        if is_malformed_span(exp_text):
            continue

        exp_start_local = min(doc[i].idx for i in exp_idxs)
        exp_end_local = max(doc[i].idx + len(doc[i].text) for i in exp_idxs)

        # Hard sentence boundary check.
        if exp_start_local < 0 or exp_end_local > len(sentence_text):
            continue

        ref_exps.append({
            "text":exp_text,
            "start_char":sentence_offset + exp_start_local,
            "end_char":sentence_offset + exp_end_local,
            "source_chunk":text,
        })

    return noun_chunks,ref_exps

# ---------------------------------------------------------------------------
# FULL CAPTION
# ---------------------------------------------------------------------------
def process_caption(nlp,caption):
    quote_spans = find_quote_spans(caption)
    sentence_spans = split_sentences(caption)
    noun_chunks = []
    ref_exps = []

    for sent_start,sent_end in sentence_spans:
        sentence_text = caption[sent_start:sent_end]
        if not sentence_text.strip():
            continue
        chunks,refs = process_sentence(
            nlp,
            sentence_text,
            sent_start,
            quote_spans,
        )
        noun_chunks.extend(chunks)
        ref_exps.extend(refs)

    noun_chunks = dedupe_contained(noun_chunks)
    ref_exps = dedupe_contained(ref_exps)

    return noun_chunks,ref_exps

# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument(
        "--in_path",
        default="/kaggle/input/datasets/hayatkarabulut/pixmo-cap-raw/01_pairs.jsonl"
    )
    ap.add_argument(
        "--out_path",
        default="/kaggle/working/noun_chunks15_70k.jsonl"
    )
    ap.add_argument(
        "--spacy_model",
        default="en_core_web_sm"
    )
    ap.add_argument(
        "--limit",
        type=int,
        default=70000
    )
    args,_ = ap.parse_known_args()

    nlp = spacy.load(args.spacy_model)

    n_in = 0
    n_out = 0
    n_dropped = 0
    n_count = 0
    with open(args.in_path,encoding="utf-8") as f_in, open(args.out_path,"w",encoding="utf-8") as f_out:
        progress = tqdm(
            f_in,
            total=70000,
            desc="Processing captions",
            unit="caption",
        )
        for line in f_in:
            if n_in!=700000:
                n_in += 1
                if n_in>630000:
                    rec = json.loads(line)
                    n_count += 1
                    noun_chunks,ref_exps = process_caption(
                        nlp,
                        rec["caption"]
                    )
        
                    if not ref_exps:
                        n_dropped += 1
                        continue
        
                    rec["noun_chunks"] = noun_chunks
                    rec["ref_exps"] = ref_exps
        
                    f_out.write(
                        json.dumps(rec,ensure_ascii=False) + "\n"
                    )
                    n_out += 1
                    progress.set_postfix(
                        kept=n_count,
                        dropped=n_dropped
                    )

        progress.close()

    print(
        f"In: {n_in}  "
        f"Out: {n_out}  "
        f"Dropped (no groundable chunks): {n_dropped}"
    )
    from IPython.display import HTML, display
    import os
    
    output_path = "/kaggle/working/noun_chunks15_70k.jsonl"
    
    display(HTML(
        f'<a href="/kaggle/working/{os.path.basename(output_path)}" download>'
        f'📥 Download {os.path.basename(output_path)}</a>'
    ))

if __name__ == "__main__":
    main()

!pip install -q torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126
!pip install -q torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126

import torch, torchvision
print(torch.__version__, torchvision.__version__)
print(torch.cuda.get_arch_list())
x = torch.randn(4, 4).cuda()
print((x @ x).is_cuda)

# after restarting the kernel/session:
!pip install -q --no-deps transformers accelerate
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

for mem leak at a certain line:

import io
import json
import os
import gc
import base64
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, as_completed

import requests
import torch
from PIL import Image
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from tqdm import tqdm
from IPython.display import HTML, display


def download_file_in_notebook(file_path: Path):
    """Triggers an automatic browser download in Kaggle or Google Colab environments."""
    if not file_path.exists():
        print(f"Error: Output file {file_path} not found.")
        return

    # Option 1: Native Google Colab API
    try:
        from google.colab import files
        files.download(str(file_path))
        print(f"Triggered Colab download for {file_path.name}")
        return
    except ImportError:
        pass

    # Option 2: JavaScript Data URI trigger for Kaggle / Standard Jupyter
    try:
        with open(file_path, "rb") as f:
            data = f.read()
        b64_str = base64.b64encode(data).decode("utf-8")
        
        js_code = f"""
        <a id="auto_download_link" href="data:application/json;base64,{b64_str}" download="{file_path.name}"></a>
        <script>
            (function() {{
                var link = document.getElementById('auto_download_link');
                if (link) {{
                    link.click();
                }}
            }})();
        </script>
        """
        display(HTML(js_code))
        print(f"Browser download prompt triggered for {file_path.name}.")
    except Exception as e:
        print(f"Automatic browser download failed: {e}")
        print(f"File remains saved at: {file_path.resolve()}")


def fetch_single_record(rec, timeout=6):
    """Fetch image with phrase truncation and resolution safety caps."""
    phrases = [r["text"] for r in rec.get("ref_exps", [])] or \
              [c["text"] for c in rec.get("noun_chunks", [])]
    
    if not phrases:
        return rec, None, phrases, "no phrases"

    phrases = phrases[:30]  # Cap max phrases per image

    url = rec.get("image_url", "")
    try:
        with requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"}) as resp:
            resp.raise_for_status()
            with Image.open(io.BytesIO(resp.content)) as raw_img:
                img = raw_img.convert("RGB")
                if min(img.size) < 32:
                    img.close()
                    return rec, None, phrases, "image too small (<32px)"
                
                max_dim = max(img.size)
                if max_dim > 2048:
                    scale = 2048 / max_dim
                    new_size = (int(img.width * scale), int(img.height * scale))
                    img = img.resize(new_size, Image.Resampling.BILINEAR)

                return rec, img, phrases, None
    except Exception as e:
        return rec, None, phrases, str(e)


def ground_image(processor, model, pil_image, phrases, device="cuda", score_threshold=0.1):
    """Runs OWLv2 inference safely."""
    w, h = pil_image.size

    inputs = processor(
        text=[phrases], images=pil_image, return_tensors="pt",
        padding=True, truncation=True,
    ).to(device)

    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([[h, w]], device=device)
    results = processor.post_process_grounded_object_detection(
        outputs=outputs, target_sizes=target_sizes, threshold=score_threshold
    )[0]

    dets = []
    for box, score, label_idx in zip(results["boxes"], results["scores"], results["labels"]):
        x1, y1, x2, y2 = box.tolist()
        idx = int(label_idx)
        dets.append({
            "phrase": phrases[idx],
            "phrase_idx": idx,
            "box_xyxy": [x1, y1, x2, y2],
            "score": float(score),
        })

    del inputs, outputs, target_sizes, results
    return dets, w, h


def main():
    input_file_path = Path('/kaggle/input/datasets/hayatkarabulut/15-02/noun_chunks15_70k.jsonl')
    out_path = Path(f"/kaggle/working/15_03.jsonl")
    
    model_id = "google/owlv2-base-patch16-ensemble"
    device = "cuda"
    score_threshold = 0.1
    
    num_workers = 2        
    MAX_QUEUE_SIZE = 4     

    # --- RESUME SYSTEM: Load existing completed IDs ---
    done_ids = set()
    if out_path.exists():
        with open(out_path, "r") as f_done:
            for line in f_done:
                try:
                    r = json.loads(line)
                    if "id" in r:
                        done_ids.add(r["id"])
                except Exception:
                    continue
        print(f"Resuming file: Found {len(done_ids)} previously completed records.")

    processor = Owlv2Processor.from_pretrained(model_id)
    model = Owlv2ForObjectDetection.from_pretrained(
        model_id, torch_dtype=torch.float16
    ).to(device)
    model.eval()

    os.makedirs(out_path.parent, exist_ok=True)
    n_in, n_ok, n_fetch_failed, n_no_phrases = 0, 0, 0, 0

    with open(input_file_path) as f_count:
        total_lines = sum(1 for _ in f_count)

    # Open in append mode ('a') to keep previous work
    with open(input_file_path) as f_in, open(out_path, "a") as f_out, ThreadPoolExecutor(max_workers=num_workers) as executor:
        pbar = tqdm(total=total_lines, desc=f"Processing {input_file_path.name}")
        pending_futures = set()

        def process_result(rec, pil_image, phrases, err):
            nonlocal n_in, n_ok, n_fetch_failed, n_no_phrases
            n_in += 1

            if err == "no phrases":
                n_no_phrases += 1
            elif pil_image is None:
                n_fetch_failed += 1
            else:
                try:
                    dets, w, h = ground_image(
                        processor, model, pil_image, phrases,
                        device=device, score_threshold=score_threshold,
                    )
                    rec["width"] = w
                    rec["height"] = h
                    rec["raw_boxes"] = dets
                    f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    f_out.flush()
                    n_ok += 1
                except Exception as e:
                    tqdm.write(f"skip {rec.get('id')}: grounding failed ({e})")
                finally:
                    pil_image.close()
                    del pil_image

            pbar.update(1)
            pbar.set_postfix(ok=n_ok, failed=n_fetch_failed)

            if n_in % 50 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        for line in f_in:
            rec = json.loads(line)
            
            # SKIP records that were already processed in a previous run
            if rec.get("id") in done_ids:
                pbar.update(1)
                n_in += 1
                continue

            fut = executor.submit(fetch_single_record, rec)
            pending_futures.add(fut)

            if len(pending_futures) >= MAX_QUEUE_SIZE:
                done, pending_futures = wait(pending_futures, return_when=FIRST_COMPLETED)
                for f in done:
                    rec_res, img_res, phrases_res, err_res = f.result()
                    process_result(rec_res, img_res, phrases_res, err_res)

        for f in as_completed(pending_futures):
            rec_res, img_res, phrases_res, err_res = f.result()
            process_result(rec_res, img_res, phrases_res, err_res)

    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"\nFinished {input_file_path.name} — Total: {n_in} | Grounded: {n_ok} | "
        f"Fetch failures: {n_fetch_failed} | No phrases: {n_no_phrases}\n"
    )

    # Automatically trigger file download in browser
    download_file_in_notebook(out_path)


if __name__ == "__main__":
    main()

import argparse
import json
import os

import torch
from torchvision.ops import nms


MIN_BOX_SIDE = 32

# Extensions considered "input files" when --in_path is a directory.
# Anything else in the directory (e.g. stray metadata files) is skipped.
INPUT_EXTS = (".jsonl", ".json")


def run_nms(dets, iou_thresh=0.5):
    if not dets:
        return []
    boxes = torch.tensor([d["box_xyxy"] for d in dets], dtype=torch.float32)
    scores = torch.tensor([d["score"] for d in dets], dtype=torch.float32)
    keep_idx = nms(boxes, scores, iou_thresh)
    return [dets[i] for i in keep_idx.tolist()]


def to_normalized(box_xyxy, w, h):
    x1, y1, x2, y2 = box_xyxy
    return [
        max(0.0, x1 / w), max(0.0, y1 / h),
        min(1.0, x2 / w), min(1.0, y2 / h),
    ]


def process_file(in_path, f_out, args):
    """Filter+assemble records from a single input file, streaming the
    survivors straight into the already-open shared output handle."""
    n_in, n_kept, n_dropped_empty = 0, 0, 0

    with open(in_path) as f_in:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            n_in += 1
            rec = json.loads(line)

            w, h = rec.get("width"), rec.get("height")
            if not w or not h:
                continue

            # same fallback logic stage 3 used to decide what to query with —
            # must match exactly, since phrase_idx indexes into THIS list.
            ref_exps = rec.get("ref_exps", [])
            noun_chunks = rec.get("noun_chunks", [])
            match_list = ref_exps if ref_exps else noun_chunks

            dets = rec.get("raw_boxes", [])

            # 1) size filter
            dets = [d for d in dets
                    if (d["box_xyxy"][2] - d["box_xyxy"][0]) >= args.min_box_side
                    and (d["box_xyxy"][3] - d["box_xyxy"][1]) >= args.min_box_side]

            # 2) confidence filter
            dets = [d for d in dets if d["score"] >= args.score_thresh]

            # 3) NMS across the whole image
            dets = run_nms(dets, iou_thresh=args.nms_iou)

            if not dets:
                n_dropped_empty += 1
                continue

            ref_exps_out = []
            for d in dets:
                idx = d.get("phrase_idx")
                if idx is None or idx >= len(match_list):
                    continue  # stale/mismatched index — skip rather than guess
                candidate = match_list[idx]

                x1, y1, x2, y2 = to_normalized(d["box_xyxy"], w, h)
                ref_exps_out.append([
                    candidate["start_char"], candidate["end_char"],
                    x1, y1, x2, y2, d["score"],
                ])

            if not ref_exps_out:
                n_dropped_empty += 1
                continue

            out_rec = {
                "id": rec["id"],
                "image_url": rec["image_url"],
                "width": w,
                "height": h,
                "caption": rec["caption"],
                # passed through unchanged from stage 2 — see point 5 in
                # the module docstring for why this isn't filtered/boxed
                "noun_chunks": noun_chunks,
                "ref_exps": ref_exps_out,
            }
            f_out.write(json.dumps(out_rec, ensure_ascii=False) + "\n")
            n_kept += 1

    return n_in, n_kept, n_dropped_empty


def collect_input_files(in_path):
    """--in_path may be a directory (process every file in it) or a single
    file (kept for backwards compatibility)."""
    if os.path.isdir(in_path):
        files = sorted(
            os.path.join(in_path, fname)
            for fname in os.listdir(in_path)
            if os.path.isfile(os.path.join(in_path, fname))
            and fname.lower().endswith(INPUT_EXTS)
        )
        if not files:
            raise SystemExit(f"No {INPUT_EXTS} files found in directory: {in_path}")
        return files
    elif os.path.isfile(in_path):
        return [in_path]
    else:
        raise SystemExit(f"--in_path does not exist: {in_path}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_path", default="/kaggle/working/15_03.jsonl",
                     help="directory of input files to process (or a single file, "
                          "for backwards compatibility). All surviving records from "
                          "every file are merged into one --out_path jsonl.")
    ap.add_argument("--out_path", default="/kaggle/working/15_04.jsonl")
    ap.add_argument("--score_thresh", type=float, default=0.3,
                     help="pick this from 03b_visualize_boxes.py's printed score "
                          "percentiles for YOUR data — do not reuse the paper's 0.65, "
                          "that was calibrated for GLIP, not OWLv2.")
    ap.add_argument("--min_box_side", type=int, default=MIN_BOX_SIDE)
    ap.add_argument("--nms_iou", type=float, default=0.5)
    args, _ = ap.parse_known_args()

    in_files = collect_input_files(args.in_path)

    total_in, total_kept, total_dropped = 0, 0, 0
    os.makedirs(os.path.dirname(args.out_path) or ".", exist_ok=True)
    with open(args.out_path, "w") as f_out:
        for in_file in in_files:
            n_in, n_kept, n_dropped_empty = process_file(in_file, f_out, args)
            total_in += n_in
            total_kept += n_kept
            total_dropped += n_dropped_empty
            print(f"  [{os.path.basename(in_file)}] In: {n_in}  "
                  f"Kept: {n_kept}  Dropped: {n_dropped_empty}")

    print(f"Processed {len(in_files)} file(s) -> {args.out_path}")
    print(f"TOTAL — In: {total_in}  Kept: {total_kept}  "
          f"Dropped (no boxes survived filtering): {total_dropped}")


if __name__ == "__main__":
    main()


"""
Build final training-ready webdataset tar shards from stage 4's output
(04_final.jsonl) — filtering (size/confidence/NMS) already happened there.

Updates for Kaggle (Smaller Batches & HyCoCLIP Sizing):
- Limit of 10 .tar files (shards) per run (configurable via --max_tars).
- Saves completed .tars directly to /kaggle/working/ (no zipping).
- Persists progress to `progress.json`.
- Starts at shard index 25 (0025.tar) when initiating fresh.
- When re-running the cell, perfectly resumes exactly where it left off.
- ALL images (main scene and crops) are resized/center-cropped to 224x224 
  to match HyCoCLIP ViT requirements and save massive disk space.
"""

import argparse
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import io
import json
import os
import shutil
import tarfile

import requests
from requests.adapters import HTTPAdapter
from PIL import Image

SAMPLES_PER_SHARD = 5000
INPUT_EXTS = (".jsonl", ".json")


# ---------------------------------------------------------------------------
# State Management (Resume Capability)
# ---------------------------------------------------------------------------
def load_progress(out_dir, default_shard_idx=64):
    prog_file = "/kaggle/working/progress.json"
    if os.path.exists(prog_file):
        with open(prog_file, "r") as f:
            return json.load(f)
    return {
        "next_line_idx": 0,
        "global_idx": 0,
        "shard_idx": default_shard_idx
    }

def save_progress(out_dir, state):
    with open(os.path.join(out_dir, "progress.json"), "w") as f:
        json.dump(state, f)


# ---------------------------------------------------------------------------
# Noun-chunk matching
# ---------------------------------------------------------------------------
def find_matching_noun_chunk(ref_start, ref_end, noun_chunks):
    candidates = [
        nc for nc in noun_chunks
        if nc["start_char"] >= ref_start and nc["end_char"] <= ref_end
    ]
    if not candidates:
        return None
    return min(candidates, key=lambda nc: nc["start_char"])


# ---------------------------------------------------------------------------
# Image fetch + crop + encode + resize
# ---------------------------------------------------------------------------
def fetch_image(url, session, timeout=10):
    try:
        resp = session.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        if min(img.size) < 32:
            img.close()
            return None, "image too small (<32px on a side)"
        return img, None
    except Exception as e:
        return None, str(e)


def denormalize_box(x1n, y1n, x2n, y2n, w, h):
    return [x1n * w, y1n * h, x2n * w, y2n * h]


def crop_box(pil_image, box_xyxy):
    w, h = pil_image.size
    x1, y1, x2, y2 = box_xyxy
    x1 = max(0, min(w - 1, int(x1)))
    y1 = max(0, min(h - 1, int(y1)))
    x2 = max(x1 + 1, min(w, int(x2)))
    y2 = max(y1 + 1, min(h, int(y2)))
    return pil_image.crop((x1, y1, x2, y2))


def process_image_to_hycoclip_size(pil_image, target_size=224):
    """
    Resizes and center crops an image to target_size x target_size, 
    matching HyCoCLIP's T.Compose([T.Resize(224, BICUBIC), T.CenterCrop(224)])
    """
    w, h = pil_image.size
    
    # 1. Resize shortest edge to target_size using Bicubic interpolation
    if w < h:
        new_w = target_size
        new_h = int(h * (target_size / w))
    else:
        new_h = target_size
        new_w = int(w * (target_size / h))
    
    # In Pillow >= 9.1.0, Image.BICUBIC is replaced by Image.Resampling.BICUBIC
    img_resized = pil_image.resize((new_w, new_h), getattr(Image, 'Resampling', Image).BICUBIC)
    
    # 2. Center crop to exact target_size x target_size
    left = (new_w - target_size) / 2
    top = (new_h - target_size) / 2
    right = (new_w + target_size) / 2
    bottom = (new_h + target_size) / 2
    
    return img_resized.crop((left, top, right, bottom))


def jpeg_bytes(pil_image, quality=92):
    buf = io.BytesIO()
    pil_image.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()


# ---------------------------------------------------------------------------
# Tar writing
# ---------------------------------------------------------------------------
def add_bytes(tar, name, data):
    info = tarfile.TarInfo(name=name)
    info.size = len(data)
    tar.addfile(info, io.BytesIO(data))


def write_sample(tar, idx, child_jpg_bytes, caption, boxes_with_text_bytes):
    prefix = f"{idx:09d}"
    add_bytes(tar, f"{prefix}.child.jpg", child_jpg_bytes)
    add_bytes(tar, f"{prefix}.child.txt", caption.encode("utf-8"))
    add_bytes(tar, f"{prefix}.numparents.txt", str(len(boxes_with_text_bytes)).encode("utf-8"))
    for i, (crop_bytes, text) in enumerate(boxes_with_text_bytes):
        add_bytes(tar, f"{prefix}.parent{i:03d}.jpg", crop_bytes)
        add_bytes(tar, f"{prefix}.parent{i:03d}.txt", text.encode("utf-8"))


# ---------------------------------------------------------------------------
# Input collection & Shard Writer
# ---------------------------------------------------------------------------
def collect_input_files(in_path):
    if os.path.isdir(in_path):
        files = sorted(
            os.path.join(in_path, fname)
            for fname in os.listdir(in_path)
            if os.path.isfile(os.path.join(in_path, fname))
            and fname.lower().endswith(INPUT_EXTS)
        )
        if not files:
            raise SystemExit(f"No {INPUT_EXTS} files found in directory: {in_path}")
        return files
    elif os.path.isfile(in_path):
        return [in_path]
    else:
        raise SystemExit(f"--in_path does not exist: {in_path}")


def iter_records(in_files, skip_lines):
    line_idx = 0
    for path in in_files:
        with open(path) as f_in:
            for line in f_in:
                if line_idx < skip_lines:
                    line_idx += 1
                    continue
                line = line.strip()
                if line:
                    yield (line_idx, json.loads(line))
                line_idx += 1


class ShardWriter:
    def __init__(self, out_dir, tmp_dir, samples_per_shard, start_shard_idx, max_tars=10):
        self.out_dir = out_dir
        self.tmp_dir = tmp_dir
        self.samples_per_shard = samples_per_shard
        self.max_tars = max_tars
        
        self.shard_idx = start_shard_idx
        self.count_in_shard = 0
        self.tar = None
        self.tmp_path = None
        self.tars_created_this_run = 0

        os.makedirs(self.tmp_dir, exist_ok=True)
        os.makedirs(self.out_dir, exist_ok=True)
        self._open_new_shard()

    def _open_new_shard(self):
        self.tmp_path = os.path.join(self.tmp_dir, f"{self.shard_idx:04d}.tar")
        print(f"  opening shard in temp dir: {self.tmp_path}")
        self.tar = tarfile.open(self.tmp_path, "w")
        self.count_in_shard = 0

    def _close_and_move_shard(self):
        """Closes the active tar and moves it to the final output directory."""
        self.tar.close()
        out_path = os.path.join(self.out_dir, f"{self.shard_idx:04d}.tar")
        shutil.move(self.tmp_path, out_path)
        print(f"  Saved completed shard directly to: {out_path}")
        
        self.tars_created_this_run += 1
        self.shard_idx += 1

    def write(self, idx, child_jpg_bytes, caption, boxes_with_text_bytes):
        write_sample(self.tar, idx, child_jpg_bytes, caption, boxes_with_text_bytes)
        self.count_in_shard += 1

        if self.count_in_shard >= self.samples_per_shard:
            self._close_and_move_shard()

            # Only open a new shard if we haven't reached our max tar limit for this run
            if self.tars_created_this_run < self.max_tars:
                self._open_new_shard()

    def flush_final(self):
        """Flushes whatever is left if the JSONL file ends entirely."""
        if self.tar and not self.tar.closed:
            if self.count_in_shard > 0:
                self._close_and_move_shard()
            else:
                self.tar.close()
                if os.path.exists(self.tmp_path):
                    os.remove(self.tmp_path)


# ---------------------------------------------------------------------------
# Worker & Pipeline setup
# ---------------------------------------------------------------------------
def process_record(rec, session):
    w, h = rec.get("width"), rec.get("height")
    caption = rec.get("caption", "")
    noun_chunks = rec.get("noun_chunks", [])
    ref_exps = rec.get("ref_exps", [])

    if not ref_exps or not w or not h:
        return "no_boxes", rec.get("id"), None, None, None

    resolved = []
    for row in ref_exps:
        start, end, x1n, y1n, x2n, y2n, score = row
        matched_nc = find_matching_noun_chunk(start, end, noun_chunks)
        text = matched_nc["text"] if matched_nc else caption[start:end]
        box_xyxy = denormalize_box(x1n, y1n, x2n, y2n, w, h)
        resolved.append((box_xyxy, text))

    if not resolved:
        return "no_boxes", rec.get("id"), None, None, None

    pil_image, err = fetch_image(rec["image_url"], session=session)
    if pil_image is None:
        return "fetch_failed", rec.get("id"), err, None, None

    # Process the main (child) image to 224x224
    processed_main = process_image_to_hycoclip_size(pil_image, target_size=224)
    child_jpg_bytes = jpeg_bytes(processed_main)
    processed_main.close()

    boxes_with_text_bytes = []
    for box_xyxy, text in resolved:
        # Crop the box from the original untouched high-res image
        crop = crop_box(pil_image, box_xyxy)
        
        # Process the crop (parent) to 224x224
        processed_crop = process_image_to_hycoclip_size(crop, target_size=224)
        crop_bytes = jpeg_bytes(processed_crop)
        boxes_with_text_bytes.append((crop_bytes, text))
        
        processed_crop.close()
        crop.close()

    pil_image.close()

    return "success", rec.get("id"), child_jpg_bytes, caption, boxes_with_text_bytes


def bounded_map(executor, func, iterable, max_in_flight):
    """Yields results in strict input order while keeping max_in_flight tasks in flight."""
    it = iter(iterable)
    futures = deque()

    for item in it:
        futures.append(executor.submit(func, item))
        if len(futures) >= max_in_flight:
            break

    while futures:
        f = futures.popleft()
        yield f.result()
        try:
            next_item = next(it)
            futures.append(executor.submit(func, next_item))
        except StopIteration:
            pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_path", default="/kaggle/input/datasets/hayatkarabulut/15-04-icin/15_04.jsonl")
    ap.add_argument("--out_dir", default="/kaggle/working")
    ap.add_argument("--samples_per_shard", type=int, default=SAMPLES_PER_SHARD)
    ap.add_argument("--max_tars", type=int, default=13, help="Maximum number of tar shards to produce in a single run")
    ap.add_argument("--start_shard_idx", type=int, default=116, help="Starting shard index if state file does not exist")
    ap.add_argument("--num_workers", type=int, default=96)
    args, _ = ap.parse_known_args()

    tmp_dir = "/tmp/webdataset_shards"
    os.makedirs(args.out_dir, exist_ok=True)
    in_files = collect_input_files(args.in_path)

    # Load State for resuming
    state = load_progress(args.out_dir, default_shard_idx=args.start_shard_idx)
    print(f"Resuming from state -> Line: {state.get('next_line_idx', 0)} | Webdataset ID: {state.get('global_idx', 0)} | Shard ID: {state.get('shard_idx', 25)}")

    writer = ShardWriter(
        args.out_dir, 
        tmp_dir, 
        args.samples_per_shard, 
        start_shard_idx=state.get("shard_idx", args.start_shard_idx),
        max_tars=args.max_tars
    )

    session = requests.Session()
    adapter = HTTPAdapter(pool_connections=args.num_workers, pool_maxsize=args.num_workers)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    global_idx = state.get("global_idx", 0)
    n_seen, n_written, n_fetch_failed, n_no_boxes = 0, 0, 0, 0

    records_gen = iter_records(in_files, skip_lines=state.get("next_line_idx", 0))

    def worker_task(arg):
        line_idx, rec = arg
        status, rec_id, res1, res2, res3 = process_record(rec, session)
        return line_idx, status, rec_id, res1, res2, res3

    with ThreadPoolExecutor(max_workers=args.num_workers) as executor:
        for res in bounded_map(executor, worker_task, records_gen, max_in_flight=args.num_workers * 2):
            line_idx, status, rec_id, res1, res2, res3 = res
            n_seen += 1

            if status == "no_boxes":
                n_no_boxes += 1
            elif status == "fetch_failed":
                n_fetch_failed += 1
            elif status == "success":
                writer.write(global_idx, res1, res2, res3)
                global_idx += 1
                n_written += 1

                if n_written % 1000 == 0:
                    print(f"  written {n_written} samples in this run ({n_fetch_failed} fetch failures)...")

            # Continuously update state so the next run knows exactly where to resume
            state["next_line_idx"] = line_idx + 1

            # STOP exactly when max_tars limit has been reached
            if writer.tars_created_this_run >= args.max_tars:
                print(f"\nTarget of {args.max_tars} tar shards reached! Stopping gracefully to prevent hitting storage limits...")
                break

    # If the JSONL ends before filling all tars, flush the remaining samples
    if writer.tars_created_this_run < args.max_tars:
        writer.flush_final()

    # Save final progress for this run
    state["global_idx"] = global_idx
    state["shard_idx"] = writer.shard_idx
    save_progress(args.out_dir, state)

    # Clean up the temp directory to keep Kaggle memory empty
    shutil.rmtree(tmp_dir, ignore_errors=True)

    print(
        f"\nRun Stats -> Seen: {n_seen}  Written: {n_written}  "
        f"Fetch failures: {n_fetch_failed}  No boxes: {n_no_boxes}"
    )


if __name__ == "__main__":
    main()

update!!!

In [ ]:
"""
Build final training-ready webdataset tar shards from a single JSONL file.
- Reads initial progress from --in_progress_path (/kaggle/input/...).
- Writes updated progress to --out_progress_path (/kaggle/working/...).
- Letterboxes images to 224x224.
"""

import argparse
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import io
import json
import os
import shutil
import tarfile

import requests
from requests.adapters import HTTPAdapter
from PIL import Image

SAMPLES_PER_SHARD = 5000


# ---------------------------------------------------------------------------
# State Management (Separate Input Read / Output Write)
# ---------------------------------------------------------------------------
def load_progress(in_progress_path, default_shard_idx=0):
    if in_progress_path and os.path.exists(in_progress_path):
        print(f"Loading initial state from INPUT: {in_progress_path}")
        with open(in_progress_path, "r") as f:
            return json.load(f)
    print("No valid input progress file found. Starting fresh.")
    return {
        "next_line_idx": 0,
        "global_idx": 0,
        "shard_idx": default_shard_idx
    }

def save_progress(out_progress_path, state):
    with open(out_progress_path, "w") as f:
        json.dump(state, f)
    print(f"Saved updated progress to OUTPUT: {out_progress_path}")


# ---------------------------------------------------------------------------
# Noun-chunk matching
# ---------------------------------------------------------------------------
def find_matching_noun_chunk(ref_start, ref_end, noun_chunks):
    candidates = [
        nc for nc in noun_chunks
        if nc["start_char"] >= ref_start and nc["end_char"] <= ref_end
    ]
    if not candidates:
        return None
    return min(candidates, key=lambda nc: nc["start_char"])


# ---------------------------------------------------------------------------
# Image fetch + crop + encode + letterbox resize
# ---------------------------------------------------------------------------
def fetch_image(url, session, timeout=10):
    try:
        resp = session.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        if min(img.size) < 32:
            img.close()
            return None, "image too small (<32px on a side)"
        return img, None
    except Exception as e:
        return None, str(e)


def denormalize_box(x1n, y1n, x2n, y2n, w, h):
    return [x1n * w, y1n * h, x2n * w, y2n * h]


def crop_box(pil_image, box_xyxy):
    w, h = pil_image.size
    x1, y1, x2, y2 = box_xyxy
    x1 = max(0, min(w - 1, int(x1)))
    y1 = max(0, min(h - 1, int(y1)))
    x2 = max(x1 + 1, min(w, int(x2)))
    y2 = max(y1 + 1, min(h, int(y2)))
    return pil_image.crop((x1, y1, x2, y2))


def process_image_to_hycoclip_size(pil_image, target_size=224, pad_color=(0, 0, 0)):
    w, h = pil_image.size
    scale = target_size / max(w, h)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    
    img_resized = pil_image.resize((new_w, new_h), getattr(Image, 'Resampling', Image).BICUBIC)
    
    new_img = Image.new("RGB", (target_size, target_size), pad_color)
    paste_x = (target_size - new_w) // 2
    paste_y = (target_size - new_h) // 2
    new_img.paste(img_resized, (paste_x, paste_y))
    
    return new_img


def jpeg_bytes(pil_image, quality=92):
    buf = io.BytesIO()
    pil_image.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()


# ---------------------------------------------------------------------------
# Tar writing
# ---------------------------------------------------------------------------
def add_bytes(tar, name, data):
    info = tarfile.TarInfo(name=name)
    info.size = len(data)
    tar.addfile(info, io.BytesIO(data))


def write_sample(tar, idx, child_jpg_bytes, caption, boxes_with_text_bytes):
    prefix = f"{idx:09d}"
    add_bytes(tar, f"{prefix}.child.jpg", child_jpg_bytes)
    add_bytes(tar, f"{prefix}.child.txt", caption.encode("utf-8"))
    add_bytes(tar, f"{prefix}.numparents.txt", str(len(boxes_with_text_bytes)).encode("utf-8"))
    for i, (crop_bytes, text) in enumerate(boxes_with_text_bytes):
        add_bytes(tar, f"{prefix}.parent{i:03d}.jpg", crop_bytes)
        add_bytes(tar, f"{prefix}.parent{i:03d}.txt", text.encode("utf-8"))


# ---------------------------------------------------------------------------
# Single-file Line Iterator & Shard Writer
# ---------------------------------------------------------------------------
def iter_records(in_path, skip_lines):
    line_idx = 0
    with open(in_path) as f_in:
        for line in f_in:
            if line_idx < skip_lines:
                line_idx += 1
                continue
            line = line.strip()
            if line:
                yield line_idx, json.loads(line)
            line_idx += 1


class ShardWriter:
    def __init__(self, out_dir, tmp_dir, samples_per_shard, start_shard_idx, max_tars=10):
        self.out_dir = out_dir
        self.tmp_dir = tmp_dir
        self.samples_per_shard = samples_per_shard
        self.max_tars = max_tars
        
        self.shard_idx = start_shard_idx
        self.count_in_shard = 0
        self.tar = None
        self.tmp_path = None
        self.tars_created_this_run = 0

        os.makedirs(self.tmp_dir, exist_ok=True)
        os.makedirs(self.out_dir, exist_ok=True)
        self._open_new_shard()

    def _open_new_shard(self):
        self.tmp_path = os.path.join(self.tmp_dir, f"{self.shard_idx:04d}.tar")
        print(f"  Opening shard in temp dir: {self.tmp_path}")
        self.tar = tarfile.open(self.tmp_path, "w")
        self.count_in_shard = 0

    def _close_and_move_shard(self):
        self.tar.close()
        out_path = os.path.join(self.out_dir, f"{self.shard_idx:04d}.tar")
        shutil.move(self.tmp_path, out_path)
        print(f"  Saved completed shard directly to: {out_path}")
        
        self.tars_created_this_run += 1
        self.shard_idx += 1

    def write(self, idx, child_jpg_bytes, caption, boxes_with_text_bytes):
        write_sample(self.tar, idx, child_jpg_bytes, caption, boxes_with_text_bytes)
        self.count_in_shard += 1

        if self.count_in_shard >= self.samples_per_shard:
            self._close_and_move_shard()
            if self.tars_created_this_run < self.max_tars:
                self._open_new_shard()

    def flush_final(self):
        if self.tar and not self.tar.closed:
            if self.count_in_shard > 0:
                self._close_and_move_shard()
            else:
                self.tar.close()
                if os.path.exists(self.tmp_path):
                    os.remove(self.tmp_path)


# ---------------------------------------------------------------------------
# Worker & Pipeline setup
# ---------------------------------------------------------------------------
def process_record(rec, session):
    w, h = rec.get("width"), rec.get("height")
    caption = rec.get("caption", "")
    noun_chunks = rec.get("noun_chunks", [])
    ref_exps = rec.get("ref_exps", [])

    if not ref_exps or not w or not h:
        return "no_boxes", rec.get("id"), None, None, None

    resolved = []
    for row in ref_exps:
        start, end, x1n, y1n, x2n, y2n, score = row
        matched_nc = find_matching_noun_chunk(start, end, noun_chunks)
        text = matched_nc["text"] if matched_nc else caption[start:end]
        box_xyxy = denormalize_box(x1n, y1n, x2n, y2n, w, h)
        resolved.append((box_xyxy, text))

    if not resolved:
        return "no_boxes", rec.get("id"), None, None, None

    pil_image, err = fetch_image(rec["image_url"], session=session)
    if pil_image is None:
        return "fetch_failed", rec.get("id"), err, None, None

    processed_main = process_image_to_hycoclip_size(pil_image, target_size=224)
    child_jpg_bytes = jpeg_bytes(processed_main)
    processed_main.close()

    boxes_with_text_bytes = []
    for box_xyxy, text in resolved:
        crop = crop_box(pil_image, box_xyxy)
        processed_crop = process_image_to_hycoclip_size(crop, target_size=224)
        crop_bytes = jpeg_bytes(processed_crop)
        boxes_with_text_bytes.append((crop_bytes, text))
        processed_crop.close()
        crop.close()

    pil_image.close()
    return "success", rec.get("id"), child_jpg_bytes, caption, boxes_with_text_bytes


def bounded_map(executor, func, iterable, max_in_flight):
    it = iter(iterable)
    futures = deque()

    for item in it:
        futures.append(executor.submit(func, item))
        if len(futures) >= max_in_flight:
            break

    while futures:
        f = futures.popleft()
        yield f.result()
        try:
            next_item = next(it)
            futures.append(executor.submit(func, next_item))
        except StopIteration:
            pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--in_path", default="/kaggle/input/datasets/hayatkarabulut/15-04-icin/15_04.jsonl")
    
    # READ progress state from input dataset
    ap.add_argument("--in_progress_path", default="/kaggle/input/datasets/hayatkarabulut/prgrs3/progress (5).json")
    # WRITE updated progress state to working directory
    ap.add_argument("--out_progress_path", default="/kaggle/working/progress.json")
    
    ap.add_argument("--out_dir", default="/kaggle/working")
    ap.add_argument("--samples_per_shard", type=int, default=SAMPLES_PER_SHARD)
    ap.add_argument("--max_tars", type=int, default=13)
    ap.add_argument("--start_shard_idx", type=int, default=116)
    ap.add_argument("--num_workers", type=int, default=96)
    args, _ = ap.parse_known_args()

    tmp_dir = "/tmp/webdataset_shards"
    os.makedirs(args.out_dir, exist_ok=True)

    # Load initial state from INPUT path
    state = load_progress(args.in_progress_path, default_shard_idx=args.start_shard_idx)
    print(f" -> Next Line: {state.get('next_line_idx', 0)} | Global ID: {state.get('global_idx', 0)} | Shard ID: {state.get('shard_idx', 0)}")

    writer = ShardWriter(
        args.out_dir, 
        tmp_dir, 
        args.samples_per_shard, 
        start_shard_idx=116,
        max_tars=args.max_tars
    )

    session = requests.Session()
    adapter = HTTPAdapter(pool_connections=args.num_workers, pool_maxsize=args.num_workers)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    global_idx = 0
    n_seen, n_written, n_fetch_failed, n_no_boxes, n_total_boxes = 0, 0, 0, 0, 0

    records_gen = iter_records(args.in_path, skip_lines=0)

    def worker_task(arg):
        line_idx, rec = arg
        status, rec_id, res1, res2, res3 = process_record(rec, session)
        return line_idx, status, rec_id, res1, res2, res3

    with ThreadPoolExecutor(max_workers=args.num_workers) as executor:
        for res in bounded_map(executor, worker_task, records_gen, max_in_flight=args.num_workers * 2):
            line_idx, status, rec_id, res1, res2, res3 = res
            n_seen += 1

            if status == "no_boxes":
                n_no_boxes += 1
            elif status == "fetch_failed":
                n_fetch_failed += 1
            elif status == "success":
                writer.write(global_idx, res1, res2, res3)
                n_total_boxes += len(res3)
                global_idx += 1
                n_written += 1

                if n_written % 1000 == 0:
                    print(f"  Written {n_written} samples in this run ({n_fetch_failed} fetch failures)...")

            state["next_line_idx"] = line_idx + 1

            if writer.tars_created_this_run >= args.max_tars:
                print(f"\nTarget of {args.max_tars} tar shards reached! Stopping gracefully...")
                break

    if writer.tars_created_this_run < args.max_tars:
        writer.flush_final()

    # Save final updated state to OUTPUT path (/kaggle/working/progress.json)
    state["global_idx"] = global_idx
    state["shard_idx"] = writer.shard_idx
    save_progress(args.out_progress_path, state)

    shutil.rmtree(tmp_dir, ignore_errors=True)

    avg_box_ratio = n_total_boxes / n_written if n_written > 0 else 0.0

    print(
        f"\nRun Stats -> Seen: {n_seen} | Written: {n_written} | "
        f"Fetch failures: {n_fetch_failed} | No boxes: {n_no_boxes}\n"
        f"Total Boxes Written: {n_total_boxes} | Average Boxes per Sample: {avg_box_ratio:.2f}"
    )


if __name__ == "__main__":
    main()

import json

# Your exact progress based on your previous logs:
# - Shards 0000 through 0011 are safely saved (12 shards * 5000 = 60,000 samples)
# - Next shard to open is 0012
# - Approximate line to skip past (60,000 written + ~551 fetch failures) = 60551

progress_data = {
    "next_line_idx": 60545,
    "global_idx": 60000,
    "shard_idx": 12
}

with open("/kaggle/working/progress.json", "w") as f:
    json.dump(progress_data, f)

print("Success! /kaggle/working/progress.json has been created.")
print("Now you can run your Stage 5 script and it will automatically pick up at 0012.tar!")

In [3]:
import json

jsonl_paths = ["/kaggle/input/datasets/hayatkarabulut/1234567-04/1_35_04 (1).jsonl","/kaggle/input/datasets/hayatkarabulut/140-210-04/8_04.jsonl","/kaggle/input/datasets/hayatkarabulut/904904/9_04.jsonl","/kaggle/input/datasets/hayatkarabulut/10-04/10_04.jsonl","/kaggle/input/datasets/hayatkarabulut/11-04-ready/11_04.jsonl","/kaggle/input/datasets/hayatkarabulut/12-04-ready/12_04.jsonl","/kaggle/input/datasets/hayatkarabulut/13-04-ready/13_04.jsonl","/kaggle/input/datasets/hayatkarabulut/14-04/14_04.jsonl","/kaggle/input/datasets/hayatkarabulut/15-04-icin/15_04.jsonl"]

total_boxes = 0
cumulative_area_ratio = 0.0

print("Calculating average bounding box area ratio...")
for jsonl in jsonl_paths:
    with open(jsonl, "r") as f:
        print("we are in: ",jsonl)
        for line in f:
            rec = json.loads(line)
            ref_exps = rec.get("ref_exps", [])
            
            for row in ref_exps:
                # row format: [start, end, x1n, y1n, x2n, y2n, score]
                _, _, x1n, y1n, x2n, y2n, _ = row
                
                # Box width and height proportions
                box_w = x2n - x1n
                box_h = y2n - y1n
                
                # Area ratio of this box compared to the full image
                area_ratio = box_w * box_h
                
                cumulative_area_ratio += area_ratio
                total_boxes += 1
    
avg_area_ratio = cumulative_area_ratio / total_boxes if total_boxes > 0 else 0.0

print("\n--- Dataset Bounding Box Area Stats ---")
print(f"Total Boxes Analyzed: {total_boxes}")
print(f"Average Box-to-Image Area Ratio: {avg_area_ratio:.4f} ({avg_area_ratio * 100:.2f}% of the image)")

Calculating average bounding box area ratio...
we are in:  /kaggle/input/datasets/hayatkarabulut/1234567-04/1_35_04 (1).jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/140-210-04/8_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/904904/9_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/10-04/10_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/11-04-ready/11_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/12-04-ready/12_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/13-04-ready/13_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/14-04/14_04.jsonl
we are in:  /kaggle/input/datasets/hayatkarabulut/15-04-icin/15_04.jsonl

--- Dataset Bounding Box Area Stats ---
Total Boxes Analyzed: 4427459
Average Box-to-Image Area Ratio: 0.1540 (15.40% of the image)


In [ ]:
import hashlib
import threading
import numpy as np
from PIL import Image, ImageStat

# ---------------------------------------------------------------------------
# 1. Color Mode & Quality Validation
# ---------------------------------------------------------------------------
def validate_color_mode(img, reject_grayscale=True, max_channel_diff=8.0):
    """
    Validates color mode and rejects single-channel / fake grayscale images.
    """
    # 1. Ensure basic mode support
    if img.mode not in ("RGB", "RGBA", "L", "P"):
        return False, f"unsupported color mode ({img.mode})"

    # 2. Check for true grayscale or near-monochrome images disguised as RGB
    if reject_grayscale:
        # If original image mode was single channel/palette
        if img.mode in ("L", "P"):
            return False, "rejected single-channel/palette grayscale image"

        # Check channel variance across R, G, B channels
        rgb_img = img.convert("RGB")
        stat = ImageStat.Stat(rgb_img)
        # Means of R, G, B channels
        means = stat.mean
        if len(means) >= 3:
            r_mean, g_mean, b_mean = means[:3]
            diff = abs(r_mean - g_mean) + abs(g_mean - b_mean) + abs(b_mean - r_mean)
            if diff < max_channel_diff:
                return False, "rejected grayscale/monochrome image in RGB container"

    return True, None


def check_visual_quality(pil_image, min_std_dev=12.0):
    """
    Filters out solid color, corrupt, low-contrast, or blank images.
    """
    arr = np.array(pil_image)
    
    # Calculate global standard deviation across pixel intensities
    std_dev = float(np.std(arr))
    if std_dev < min_std_dev:
        return False, f"low contrast/solid color image (std_dev={std_dev:.1f})"

    return True, None


# ---------------------------------------------------------------------------
# 2. Thread-Safe Deduplication (Difference Hashing / dHash)
# ---------------------------------------------------------------------------
SEEN_HASHES = set()
HASH_LOCK = threading.Lock()

def compute_dhash(pil_image, hash_size=8):
    """
    Computes a fast 64-bit Difference Hash (dHash) for perceptual deduplication.
    """
    # Convert to grayscale and resize to (hash_size + 1, hash_size)
    resized = pil_image.convert("L").resize(
        (hash_size + 1, hash_size), getattr(Image, 'Resampling', Image).BILINEAR
    )
    pixels = np.asarray(resized)
    
    # Compare adjacent pixels horizontally
    diff = pixels[:, 1:] > pixels[:, :-1]
    
    # Convert boolean matrix to integer hash string
    return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])


def is_duplicate(pil_image, max_distance=0):
    """
    Thread-safe deduplication check using dHash.
    max_distance = 0 checks for exact visual match.
    max_distance > 0 allows near-duplicate detection (e.g. max_distance=2).
    """
    img_hash = compute_dhash(pil_image)
    
    with HASH_LOCK:
        if max_distance == 0:
            if img_hash in SEEN_HASHES:
                return True
            SEEN_HASHES.add(img_hash)
            return False
        else:
            # Hamming distance comparison for near-duplicates
            for h in SEEN_HASHES:
                if bin(img_hash ^ h).count("1") <= max_distance:
                    return True
            SEEN_HASHES.add(img_hash)
            return False


# ---------------------------------------------------------------------------
# 3. Text Relevance Filtering (Heuristic or CLIP)
# ---------------------------------------------------------------------------
def check_caption_relevance(caption, min_length=10):
    """
    Fast rule-based text relevance pre-filter.
    """
    if not caption or len(caption.strip()) < min_length:
        return False, "caption too short or empty"

    # Filter generic placeholder text
    placeholders = ["stock photo", "no caption", "image placeholder", "404", "undefined"]
    if any(p in caption.lower() for p in placeholders):
        return False, "generic/placeholder caption detected"

    return True, None